# CNN-GNN-HMER CNN-GNN - Kaggle 2xT4 + MLflow (ngrok/DagsHub)

Notebook này được dựng lại dựa trên `template.ipynb` gốc để tích hợp MLflow (bắn log và checkpoint trực tiếp về máy local hoặc DagsHub):

- Giữ flow Miniconda → env `tamer` Python 3.7 → clone repo → `%cd` vào đúng project con → install → unzip CROHME → train.
- Tích hợp **MLflow** thay cho WandB để theo dõi metrics và đồng bộ model checkpoints về máy local qua ngrok hoặc DagsHub.
- Train mặc định bằng 2 GPU T4: `--trainer.gpus=2`.


## 0. Tham số chính

Nếu Kaggle không bật 2 GPU, sửa `GPUS = 1`. Còn mặc định cho 2xT4 là `GPUS = 2`.

In [ ]:
# Tham số chính cho notebook
GPUS = 2
CONFIG = "config/crohme.yaml"
WORKDIR = "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn"

# Cấu hình MLflow (ngrok hoặc DagsHub)
MLFLOW_TRACKING_URI = "https://xxxx-xxxx.ngrok-free.app"  # URL ngrok hoặc DagsHub MLflow URL
MLFLOW_TRACKING_USERNAME = ""                         # Điền username nếu dùng DagsHub
MLFLOW_TRACKING_PASSWORD = ""                         # Điền token/password nếu dùng DagsHub

print("WORKDIR:", WORKDIR)
print("CONFIG:", CONFIG)
print("GPUS:", GPUS)
print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)


## 1. Cài đặt Miniconda

In [ ]:
# Cài đặt Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Thêm conda vào PATH
import os
os.environ['PATH'] = "/kaggle/working/miniconda/bin:" + os.environ['PATH']

## 2. Tạo môi trường Python 3.7

In [ ]:
# Accept Anaconda Terms of Service cho 2 channel mặc định
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
# Tạo môi trường Python 3.7
!/kaggle/working/miniconda/bin/conda create -n tamer python=3.7 -y

## 3. Kiểm tra phiên bản Python và pip

In [ ]:
# Kiểm tra phiên bản Python và pip
shell_script = """
source /kaggle/working/miniconda/bin/activate tamer
python --version
pip --version
"""
with open("activate_env.sh", "w") as f:
    f.write(shell_script)
!bash activate_env.sh

## 4. Clone repo CNN-GNN-HMER

In [ ]:
# Clone repo
# Nếu notebook bị restart và repo đã tồn tại, cell này sẽ bỏ qua clone để không lỗi.
!if [ ! -d "/kaggle/working/CNN-GNN-HMER" ]; then git clone https://github.com/KhaiHASO/CNN-GNN-HMER.git /kaggle/working/CNN-GNN-HMER; else echo "Repo đã tồn tại: /kaggle/working/CNN-GNN-HMER"; fi

## 5. Di chuyển vào thư mục CNN-GNN

In [ ]:
# Di chuyển vào thư mục dự án con
%cd /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn
!pwd
!ls -lh

## 6. Cài đặt các gói từ conda

In [ ]:
# Cài đặt các gói từ conda
!source /kaggle/working/miniconda/bin/activate tamer && conda install pytorch-lightning=1.4.9 torchmetrics=0.6.0 -c conda-forge -y
!source /kaggle/working/miniconda/bin/activate tamer && conda install pandoc=1.19.2.1 -c conda-forge -y

## 7. Fix lỗi GLIBCXX

In [ ]:
# Cài đặt libstdcxx-ng để fix lỗi GLIBCXX
!source /kaggle/working/miniconda/bin/activate tamer && conda install -c conda-forge libstdcxx-ng -y

## 8. Cài requirements, setup.py và W&B

In [ ]:
# Cài đặt các gói từ requirements.txt và setup.py
!source /kaggle/working/miniconda/bin/activate tamer && pip install -r requirements.txt && pip install -e .

# Đảm bảo có mlflow để MLFlowLogger hoạt động
!source /kaggle/working/miniconda/bin/activate tamer && pip install mlflow


## 9. Ghi đè config/crohme.yaml chuẩn cho notebook này

In [ ]:
# Backup config cũ trước khi ghi đè
!mkdir -p config_backup
!cp config/crohme.yaml config_backup/crohme.original.yaml || true

In [ ]:
%%writefile config/crohme.yaml
seed_everything: 7
trainer:
  checkpoint_callback: true
  logger:
    class_path: pytorch_lightning.loggers.MLFlowLogger
    init_args:
      experiment_name: cnn-gnn-hmer-cnn-gnn
      tracking_uri: "http://localhost:5000"  # Sẽ được ghi đè bằng lệnh chạy CLI
      log_model: true
  callbacks:
    - class_path: pytorch_lightning.callbacks.LearningRateMonitor
      init_args:
        logging_interval: epoch
    - class_path: pytorch_lightning.callbacks.ModelCheckpoint
      init_args:
        save_top_k: 1
        monitor: val_ExpRate
        mode: max
        filename: '{epoch}-{step}-{val_ExpRate:.4f}'
  gpus: 2
  accelerator: auto
  check_val_every_n_epoch: 2
  max_epochs: 100
  deterministic: true
  precision: 16
model:
  d_model: 256
  # encoder
  growth_rate: 24
  num_layers: 16
  # decoder
  nhead: 8
  num_decoder_layers: 3
  dim_feedforward: 1024
  dc: 32
  dropout: 0.3
  vocab_size: 113  # 110 + 3
  cross_coverage: true
  self_coverage: true
  # GAT (Graph Attention Network) - optional
  use_gat: true  # CNN-GNN variant: enable GAT layers
  gat_num_layers: 2
  gat_num_heads: 8
  gat_hidden_dim: null  # null means use d_model
  gat_dropout: 0.1
  # beam search
  beam_size: 10
  max_len: 150
  alpha: 1.0
  early_stopping: false
  temperature: 1.0
  # training
  learning_rate: 1.0
  patience: 20
  milestones:
    - 300
    - 350
data:
  folder: data/crohme
  test_folder: 2014
  max_size: 320000
  scale_to_limit: true
  train_batch_size: 8
  eval_batch_size: 2
  num_workers: 5
  scale_aug: false


In [ ]:
# Kiểm tra lại config đang dùng
!echo "===== config/crohme.yaml ====="
!cat config/crohme.yaml

## 10. Cấu hình W&B riêng cho notebook này

In [ ]:
# Cấu hình các biến môi trường cho MLflow
import os
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD


In [ ]:
# Không cần login WandB nữa, kiểm tra cấu hình MLflow
print("MLflow tracking is configured to:", MLFLOW_TRACKING_URI)


## 11. Chuẩn bị dữ liệu CROHME

Config đọc dữ liệu ở `data/crohme`, nên cell này giải nén `CROHME.zip` từ repo root vào thư mục model hiện tại.

In [ ]:
# Cách 1: Giải nén CROHME.zip có sẵn trong repo hiện tại
!mkdir -p data
!if [ -f "/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip" ]; then     unzip -o /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip -d data/;   else     echo "Không thấy /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/data/CROHME.zip";     echo "Nếu dùng Kaggle Dataset, chạy cell Cách 2 bên dưới.";   fi

!echo "===== data tree ====="
!find data -maxdepth 3 -type d | sort | head -50

In [ ]:
# Cách 2: Sử dụng từ Kaggle dataset nếu anh upload CROHME.zip riêng
# Sửa path /kaggle/input/crohme-dataset/CROHME.zip nếu dataset của anh có tên khác.
# !mkdir -p data
# !cp /kaggle/input/crohme-dataset/CROHME.zip data/
# !unzip -o data/CROHME.zip -d data/
# !find data -maxdepth 3 -type d | sort | head -50

## 12. Check nhanh repo, data, config, GPU trước khi train

In [ ]:
!echo "===== PWD ====="
!pwd
!echo "===== Repo files ====="
!ls -lh
!echo "===== Config files ====="
!ls -lh config
!echo "===== Data files ====="
!find data -maxdepth 3 | head -80
!echo "===== Eval files ====="
!ls -lh eval || true

In [ ]:
# Kiểm tra CUDA và số GPU Kaggle cấp
import sys
import torch

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 13. Chạy training

In [ ]:
# Chạy training với cấu hình CROHME, truyền trực tiếp tracking_uri để log về máy local
!source /kaggle/working/miniconda/bin/activate tamer && python train.py --config {CONFIG} --trainer.gpus={GPUS} --trainer.logger.init_args.tracking_uri={MLFLOW_TRACKING_URI}


## 14. Resume training từ checkpoint nếu cần

In [ ]:
# Cell mẫu resume checkpoint từ MLflow/Kaggle.
# !source /kaggle/working/miniconda/bin/activate tamer && python train.py --config config/crohme.yaml --trainer.resume_from_checkpoint=/kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints/YOUR_CHECKPOINT.ckpt --trainer.gpus=2 --trainer.logger.init_args.tracking_uri={MLFLOW_TRACKING_URI}


In [ ]:
# Cell mẫu copy checkpoint từ Kaggle input nếu cần resume.
!mkdir -p /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints
# !cp /kaggle/input/checkpoint/YOUR_CHECKPOINT.ckpt /kaggle/working/CNN-GNN-HMER/chuyende_tamer_temp/1-cnn-gnn/lightning_logs/version_0/checkpoints/

## 15. Tìm checkpoint sau training

In [ ]:
# Liệt kê checkpoint sau training
!echo "===== Checkpoints ====="
!find lightning_logs -name "*.ckpt" -type f | sort || true

# Ghi checkpoint cuối cùng tìm được ra best_ckpt.txt để tiện eval/upload
!BEST_CKPT=$(find lightning_logs -name "*.ckpt" -type f | sort | tail -n 1); echo "$BEST_CKPT" | tee best_ckpt.txt
!echo "BEST_CKPT file content:" && cat best_ckpt.txt

## 16. Eval sau training

In [ ]:
# Eval bằng script shell có sẵn trong repo.
# Nếu eval_crohme.sh đã được cấu hình đúng trong project, cell này sẽ chạy trực tiếp.
!source /kaggle/working/miniconda/bin/activate tamer && bash eval/eval_crohme.sh

In [ ]:
# Phương án eval trực tiếp bằng eval/test.py nếu cần truyền checkpoint thủ công.
# Mặc định để comment vì cú pháp tham số phụ thuộc test.py của repo.
# BEST_CKPT=$(cat best_ckpt.txt)
# !source /kaggle/working/miniconda/bin/activate tamer && python eval/test.py --config config/crohme.yaml --checkpoint "$BEST_CKPT"

## 17. Đánh giá & Hoàn tất Đồng bộ MLflow

Do đã sử dụng MLflow, các tệp checkpoint và logs được tự động gửi thẳng về máy local / DagsHub của bạn trong suốt quá trình train. Dưới đây chỉ là hiển thị thông báo kết thúc.

In [ ]:
print("Training complete! Checkpoints are saved on your local machine / DagsHub.")


In [ ]:
print("No offline sync needed. MLflow streams metrics in real-time.")


## 18. Lưu kết quả và môi trường giống template

In [ ]:
# Lưu kết quả training/eval/config để dùng cho phiên sau
!mkdir -p /kaggle/output/cnn_gnn_results
!cp -r lightning_logs /kaggle/output/cnn_gnn_results/ || true
!cp -r config /kaggle/output/cnn_gnn_results/ || true
!cp best_ckpt.txt /kaggle/output/cnn_gnn_results/ || true
!cp -r eval /kaggle/output/cnn_gnn_results/eval_files_snapshot || true
!find /kaggle/output/cnn_gnn_results -maxdepth 3 | head -100

## 19. Nén kết quả ra /kaggle/working

In [ ]:
# Nén thư mục kết quả thành file tar.gz
!tar -czvf cnn_gnn_results.tar.gz /kaggle/output/cnn_gnn_results

# Chép file nén vào thư mục working
!cp cnn_gnn_results.tar.gz /kaggle/working/

# Xác nhận file đã được chép thành công
!ls -lh /kaggle/working/cnn_gnn_results.tar.gz